# RTSS 2026 artifact: reviewer rerun figures

This notebook loads experiment outputs generated by the reviewer and displays Figures 2-11 for *Real-Time Multi-Telescope Scheduling to Search for Transient Astrophysical Phenomena* in manuscript order.

The input CSVs are read only from `results/`. Static comparison figures are written as both PDF and PNG under `results/paper_figures/`.

Create and activate the supplied environment before opening the notebook:

```bash
conda env create -f environment.yml
conda activate rtss26-figures
jupyter lab results/RTSS2026_Rerun_Results.ipynb
```

Notes:

- Run Jupyter from the repository root, or from any directory below it. The notebook searches upward for directories named `data/` and `results/`.
- Run one or more scripts under `results/` before executing this notebook. Missing CSVs or configurations are reported without stopping the remaining figures.
- Partial experiments are supported: cross-instance and sweep plots use the available rows, so their axes may contain fewer maps or parameter values than the paper.
- When `RUN_INTERACTIVE = True`, Figures 4 and 10 create a separate full-sky Plotly view for every method. Interactive rendering can be slow, especially for maps with many likelihood points.


In [ ]:
from __future__ import annotations

import csv
import math
import re
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import Markdown, display
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Patch
from matplotlib_inline.backend_inline import set_matplotlib_formats

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")
set_matplotlib_formats("retina")


def find_repository_root(start: Path | None = None) -> Path:
    """Find the nearest parent containing both artifact data directories."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if ((candidate / "data").is_dir()
                and (candidate / "results").is_dir()):
            return candidate
    # Keep the notebook editable even before the repository is populated.
    return start


REPO_ROOT = find_repository_root()
DATA_DIR = REPO_ROOT / "data"
RESULT_DIR = REPO_ROOT / "results"
MAXP_DIR = RESULT_DIR / "max_probability"
MINK_DIR = RESULT_DIR / "min_telescope"
FIGURE_DIR = RESULT_DIR / "paper_figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Set True to build optional Plotly route viewers in addition to the
# static paper figures. The viewers are not required by the manuscript.
RUN_INTERACTIVE = False
DRAW_ALL_TILES = True
SAVE_PDF = True
SAVE_PNG = True


def first_existing(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def find_named_file(preferred: Path, name: str) -> Path:
    """Use the expected path, with a repository-wide filename fallback."""
    if preferred.exists():
        return preferred
    matches = sorted(REPO_ROOT.rglob(name), key=lambda p: (len(p.parts), str(p)))
    return matches[0] if matches else preferred


FILES = {
    # Fixed-K cross-instance and parameter-sweep results.
    "maxp_small": MAXP_DIR / "maxp_small.csv",
    "maxp_large": MAXP_DIR / "maxp_large.csv",
    "maxp_large_budget": MAXP_DIR / "maxp_large_map_budget.csv",
    "maxp_large_npath": first_existing(
        MAXP_DIR / "maxp_large_map_path.csv",
        MAXP_DIR / "maxp_large_path.csv",
    ),
    "maxp_very_large_budget": MAXP_DIR / "maxp_very_large_map_budget.csv",

    # Minimum-telescope cross-instance and deadline-sweep results.
    "mink_small": MINK_DIR / "mink_small.csv",
    "mink_large": MINK_DIR / "mink_large.csv",
    "mink_large_budget": MINK_DIR / "mink_large_budget.csv",

    # Route-visualization inputs.
    "tiling_6p9": find_named_file(
        DATA_DIR / "tilings" / "6.9x6.9_tiling.csv",
        "6.9x6.9_tiling.csv",
    ),
    "map_200220_199": find_named_file(
        DATA_DIR / "small_maps_6.9x6.9_tiling"
        / "GW200220_061928_199.txt",
        "GW200220_061928_199.txt",
    ),
    "map_200216_129": find_named_file(
        DATA_DIR / "small_maps_6.9x6.9_tiling"
        / "GW200216_220804_129.txt",
        "GW200216_220804_129.txt",
    ),
}

DATASET_200220 = "GW200220_061928_199.txt"
DATASET_200216 = "GW200216_220804_129.txt"
DATASET_LARGE = "GW191113_071753_952.txt"
DATASET_VERY_LARGE = "GW200105_162426_11678.txt"

print(f"Repository root: {REPO_ROOT}")
print(f"Figure output:   {FIGURE_DIR}")


## Input preflight

This table lists every file used by the reviewer-rerun figures. A missing row does not stop the notebook; it identifies the experiment script that must be run or the path that must be corrected in the configuration cell.


In [ ]:
preflight = pd.DataFrame(
    [
        {
            "key": key,
            "exists": path.exists(),
            "path": str(path.relative_to(REPO_ROOT))
                    if path.is_relative_to(REPO_ROOT) else str(path),
        }
        for key, path in FILES.items()
    ]
)
display(preflight)

missing = preflight.loc[~preflight["exists"], "key"].tolist()
if missing:
    display(Markdown(
        "> **Preflight:** missing inputs: `" + "`, `".join(missing) +
        "`. Figures depending on them will be skipped."
    ))
else:
    display(Markdown("> **Preflight:** all required inputs are available."))


## Shared plotting style, labels, and data loaders


In [ ]:
COL_W = 3.5
DCOL_W = 7.16

MAXP_PALETTE = {
    "greedy+annealing": "#377EB8",
    "greedy+ilp": "#4DAF4A",
    "top_annealing": "#FF7F00",
    "top_ILP": "#E41A1C",
    "top_greedy": "#984EA3",
}
MAXP_METHODS = [
    "greedy+annealing", "greedy+ilp", "top_annealing", "top_ILP"
]
# Figures 2 and 3 additionally compare against the TilePy-style baseline.
MAXP_FIXED_METHODS = [*MAXP_METHODS, "top_greedy"]
MAXP_LABEL = {
    "greedy+annealing": "Greedy+SA",
    "greedy+ilp": "Greedy+ILP",
    "top_annealing": "TOP-SA",
    "top_ILP": "TOP-ILP",
    "top_greedy": "TilePy",
}
MAXP_MARKER = {
    "greedy+annealing": "s",
    "greedy+ilp": "D",
    "top_annealing": "^",
    "top_ILP": "P",
    "top_greedy": ".",
}
MAXP_ILP = {"greedy+ilp", "top_ILP"}

MINK_PALETTE = {
    "mink_rooted_greedy": "#377EB8",
    "mink_rooted_ILP": "#4DAF4A",
    "mink_rooted_annealing": "#E41A1C",
}
MINK_METHODS = [
    "mink_rooted_greedy", "mink_rooted_ILP", "mink_rooted_annealing"
]
MINK_LABEL = {
    "mink_rooted_greedy": "R-Greedy",
    "mink_rooted_ILP": "R-ILP",
    "mink_rooted_annealing": "R-SA",
}
MINK_MARKER = {
    "mink_rooted_greedy": "v",
    "mink_rooted_ILP": "P",
    "mink_rooted_annealing": "^",
}

ROUTE_COLORS = [
    "#E41A1C", "#808000", "#984EA3", "#FF7F00", "#FFD92F",
    "#A65628", "#F781BF", "#4DAF4A", "#000000", "#1E90FF",
    "#00CED1", "#377EB8",
]


def set_rtss_style() -> None:
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 10,
        "axes.labelsize": 10,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "axes.linewidth": 0.6,
        "grid.linewidth": 0.4,
        "lines.linewidth": 1.1,
        "lines.markersize": 4.0,
        "legend.frameon": True,
        "legend.framealpha": 0.90,
        "legend.edgecolor": "0.7",
        "figure.dpi": 150,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.02,
    })


set_rtss_style()


def valid_mask(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series
    return series.astype(str).str.strip().str.lower().eq("true")


def load_maxp(path: Path) -> pd.DataFrame:
    """Load and aggregate a maximum-probability benchmark CSV."""
    required = [
        "Method", "Dataset", "Budget", "nPath", "mapTiles",
        "SumProb", "SumProbBound", "TilingTime", "PlanningTime",
        "IsValid",
    ]
    df = pd.read_csv(path, usecols=required, engine="python",
                     on_bad_lines="skip")
    df = df[valid_mask(df["IsValid"])].copy()
    for column in ["Budget", "nPath", "mapTiles", "SumProb",
                   "SumProbBound", "TilingTime", "PlanningTime"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna(subset=[
        "Budget", "nPath", "mapTiles", "SumProb", "SumProbBound",
        "PlanningTime",
    ])
    df[["Budget", "nPath", "mapTiles"]] = (
        df[["Budget", "nPath", "mapTiles"]].astype(int)
    )
    return (
        df.groupby(["Method", "Dataset", "Budget", "nPath"],
                   as_index=False)
          .agg(
              SumProb=("SumProb", "max"),
              SumProbBound=("SumProbBound", "max"),
              PlanningTime=("PlanningTime", "mean"),
              TilingTime=("TilingTime", "mean"),
              mapTiles=("mapTiles", "first"),
          )
    )


def load_mink(path: Path, anomaly_ratio: float = 2.0) -> pd.DataFrame:
    """Load and aggregate a minimum-telescope benchmark CSV."""
    columns = [
        "Method", "Dataset", "Budget", "w_max", "w_acc", "nTiles",
        "CoveredTiles", "nPath", "nPathBound", "TilingTime",
        "PlanningTime", "IsValid",
    ]
    # Read by position because result rows may contain unnamed trailing paths.
    df = pd.read_csv(
        path, header=None, skiprows=1, names=columns,
        usecols=range(len(columns)), engine="python", on_bad_lines="skip",
    )
    for column in ["Budget", "nTiles", "CoveredTiles", "nPath",
                   "nPathBound", "w_max", "w_acc", "TilingTime",
                   "PlanningTime"]:
        df[column] = pd.to_numeric(df[column], errors="coerce")
    df = df.dropna(subset=["Budget", "nPath", "nPathBound", "PlanningTime"])
    for column in ["Budget", "nTiles", "CoveredTiles", "nPath", "nPathBound"]:
        df[column] = df[column].astype(int)

    is_ilp = df["Method"].eq("mink_rooted_ILP")
    df = df[~(is_ilp & ~valid_mask(df["IsValid"]))].copy()
    is_ilp = df["Method"].eq("mink_rooted_ILP")
    anomaly = (
        is_ilp & df["nPathBound"].gt(0)
        & df["nPath"].gt(anomaly_ratio * df["nPathBound"])
    )
    if anomaly.any():
        print(f"[load_mink] dropped {int(anomaly.sum())} anomalous ILP row(s)")
        df = df[~anomaly]
    if df.empty:
        raise RuntimeError(f"No valid rows remain in {path}")
    return (
        df.groupby(["Method", "Dataset", "Budget"], as_index=False)
          .agg(
              nPath=("nPath", "min"),
              nPathBound=("nPathBound", "max"),
              PlanningTime=("PlanningTime", "mean"),
              TilingTime=("TilingTime", "mean"),
              nTiles=("nTiles", "first"),
          )
    )


def normalize_dataset(name: str) -> str:
    return str(name).replace("GW", "").replace(".txt", "").strip()


def resolve_dataset(df: pd.DataFrame, requested: str) -> str:
    available = [str(value) for value in df["Dataset"].unique()]
    if requested in available:
        return requested
    target = normalize_dataset(requested)
    matches = [value for value in available
               if normalize_dataset(value) == target]
    if len(matches) != 1:
        raise ValueError(
            f"Dataset {requested!r} did not resolve uniquely. "
            f"Available datasets: {sorted(available)}"
        )
    return matches[0]


def short_maxp_dataset(name: str, tiles: int | None = None) -> str:
    head = str(name).replace("GW", "").replace(".txt", "").split("_")[0]
    return f"{head}({tiles})" if tiles is not None else head


_TILE_RE = re.compile(r"_(\d+)\.txt$|_(\d+)$")


def tile_count_from_name(name: str) -> int:
    match = _TILE_RE.search(str(name))
    return int(match.group(1) or match.group(2)) if match else 0


def short_mink_dataset(name: str) -> str:
    value = str(name).replace("GW", "").replace(".txt", "")
    parts = value.split("_")
    return f"{parts[0]}({parts[-1]})" if len(parts) >= 3 else value[:12]


## Shared figure builders


In [ ]:
def save_and_display(fig: plt.Figure, stem: str) -> list[Path]:
    """Save one rerun figure and display it in the notebook."""
    outputs = []
    if SAVE_PDF:
        path = FIGURE_DIR / f"{stem}.pdf"
        fig.savefig(path)
        outputs.append(path)
    if SAVE_PNG:
        path = FIGURE_DIR / f"{stem}.png"
        fig.savefig(path, dpi=300)
        outputs.append(path)
    display(fig)
    plt.close(fig)
    print("Wrote:", ", ".join(str(path) for path in outputs))
    return outputs


def build_or_skip(
    number: int,
    stem: str,
    required: list[Path],
    builder,
):
    missing_paths = [path for path in required if not path.exists()]
    if missing_paths:
        display(Markdown(
            f"> **Figure {number} skipped.** Missing: "
            + ", ".join(f"`{path}`" for path in missing_paths)
        ))
        return None
    try:
        fig = builder()
    except Exception as exc:
        display(Markdown(
            f"> **Figure {number} could not be generated:** "
            f"`{type(exc).__name__}: {exc}`"
        ))
        return None
    save_and_display(fig, stem)
    return None


def unique_legend(ax: plt.Axes, **kwargs) -> None:
    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    if by_label:
        ax.legend(by_label.values(), by_label.keys(), **kwargs)


def maxp_fixed_panel(ax: plt.Axes, df: pd.DataFrame, n_path: int,
                     budget: int, metric: str) -> None:
    sub = df[(df["nPath"] == n_path) & (df["Budget"] == budget)].copy()
    if sub.empty:
        raise ValueError(f"No maximum-probability rows for K={n_path}, D={budget}")
    datasets = (
        sub.groupby("Dataset")["mapTiles"].first().sort_values().index.tolist()
    )
    tiles = dict(sub.groupby("Dataset")["mapTiles"].first())
    x = np.arange(len(datasets))

    if metric == "probability":
        width = 0.84 / len(MAXP_FIXED_METHODS)
        for index, method in enumerate(MAXP_FIXED_METHODS):
            offsets = x - 0.42 + (index + 0.5) * width
            values, bounds = [], []
            for dataset in datasets:
                row = sub[(sub["Dataset"] == dataset)
                          & (sub["Method"] == method)]
                values.append(float(row["SumProb"].iloc[0])
                              if not row.empty else np.nan)
                bounds.append(float(row["SumProbBound"].iloc[0])
                              if not row.empty else np.nan)
            if not np.any(np.isfinite(values)):
                continue
            ax.bar(offsets, values, width, color=MAXP_PALETTE[method],
                   edgecolor="black", linewidth=0.25,
                   label=MAXP_LABEL[method])
            if method in MAXP_ILP:
                for xpos, lower, upper in zip(offsets, values, bounds):
                    if (np.isfinite(lower) and np.isfinite(upper)
                            and upper > lower + 1e-8):
                        ax.bar(xpos, upper - lower, width, bottom=lower,
                               facecolor="white",
                               edgecolor=MAXP_PALETTE[method],
                               hatch="////", linewidth=0.45)
        ax.set_ylabel("Detection probability")
        ymax = max(sub["SumProb"].max(), sub["SumProbBound"].max())
        ax.set_ylim(0, min(1.02, 1.05 * ymax))
        ax.grid(axis="y", linestyle=":", alpha=0.55)
    elif metric == "runtime":
        styles = ["-", "--", ":", "-."]
        for index, method in enumerate(MAXP_FIXED_METHODS):
            values = []
            for dataset in datasets:
                row = sub[(sub["Dataset"] == dataset)
                          & (sub["Method"] == method)]
                values.append(float(row["PlanningTime"].iloc[0])
                              if not row.empty else np.nan)
            if not np.any(np.isfinite(values)):
                continue
            ax.plot(x, values, marker=MAXP_MARKER[method],
                    color=MAXP_PALETTE[method], linestyle=styles[index % len(styles)],
                    label=MAXP_LABEL[method])
        ax.set_yscale("log")
        ax.set_ylabel("Planning time (s)")
        ax.axhline(3600, color="grey", linestyle="--", linewidth=0.6)
        ax.text(x[-1], 3600, "1 h timeout", color="grey", fontsize=7,
                va="bottom", ha="right")
        ax.grid(True, which="both", linestyle=":", alpha=0.55)
    else:
        raise ValueError(metric)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [short_maxp_dataset(dataset, int(tiles[dataset]))
         for dataset in datasets],
        rotation=25, ha="right",
    )
    ax.set_xlabel("GW event (tile count)")
    ax.set_axisbelow(True)


def maxp_cross_instance_figure(
    small_path: Path, large_path: Path, metric: str
) -> plt.Figure:
    small = load_maxp(small_path)
    large = load_maxp(large_path)
    fig, axes = plt.subplots(2, 1, figsize=(DCOL_W, 4.55))
    maxp_fixed_panel(axes[0], small, n_path=4, budget=100, metric=metric)
    maxp_fixed_panel(axes[1], large, n_path=4, budget=100, metric=metric)
    axes[0].text(0.01, 0.96, "Small maps", transform=axes[0].transAxes,
                 va="top", ha="left", fontsize=8)
    axes[1].text(0.01, 0.96, "Large maps", transform=axes[1].transAxes,
                 va="top", ha="left", fontsize=8)
    legend_by_label = {}
    for axis in axes:
        axis_handles, axis_labels = axis.get_legend_handles_labels()
        for handle, label in zip(axis_handles, axis_labels):
            legend_by_label.setdefault(label, handle)
    labels = list(legend_by_label)
    handles = [legend_by_label[label] for label in labels]
    if metric == "probability":
        handles.append(Patch(facecolor="white", edgecolor="black",
                             hatch="////", linewidth=0.5))
        labels.append("ILP bound")
    fig.legend(handles, labels, loc="lower center",
               bbox_to_anchor=(0.5, 0.005), ncol=len(labels), fontsize=8)
    fig.tight_layout(rect=(0, 0.11, 1, 1), h_pad=0.7)
    return fig


def _maxp_sweep_panel(ax: plt.Axes, df: pd.DataFrame, dataset: str,
                      fixed_column: str, fixed_value: int,
                      x_column: str, metric: str,
                      methods: list[str]) -> None:
    dataset = resolve_dataset(df, dataset)
    sub = df[(df["Dataset"] == dataset)
             & (df[fixed_column] == fixed_value)].copy()
    if sub.empty:
        raise ValueError(
            f"No rows for {dataset}, {fixed_column}={fixed_value}"
        )
    x_values = sorted(sub[x_column].unique())
    for method in methods:
        method_rows = sub[sub["Method"] == method]
        xs, ys, bounds = [], [], []
        for value in x_values:
            row = method_rows[method_rows[x_column] == value]
            if row.empty:
                continue
            xs.append(value)
            if metric == "probability":
                ys.append(float(row["SumProb"].iloc[0]))
                bounds.append(float(row["SumProbBound"].iloc[0]))
            else:
                ys.append(float(row["PlanningTime"].iloc[0]))
        if not xs:
            continue
        ax.plot(xs, ys, marker=MAXP_MARKER[method],
                color=MAXP_PALETTE[method], label=MAXP_LABEL[method])
        if metric == "probability" and method in MAXP_ILP:
            ax.plot(xs, bounds, linestyle="--", marker="",
                    color=MAXP_PALETTE[method], linewidth=0.9,
                    label=f"{MAXP_LABEL[method]} bound")
    if metric == "runtime":
        ax.set_yscale("log")
        ax.set_ylabel("Planning time (s)")
    else:
        ax.set_ylabel("Detection probability")
        ax.set_ylim(bottom=0)
    ax.set_xticks(x_values)
    ax.set_xlabel("Deadline $D$ (s)" if x_column == "Budget"
                  else "Number of telescopes $K$")
    ax.grid(True, which="both", linestyle=":", alpha=0.55)
    ax.set_axisbelow(True)
    unique_legend(ax, loc="best", fontsize=7)


def maxp_budget_figure(path: Path, dataset: str, n_path: int = 4,
                      methods: list[str] | None = None) -> plt.Figure:
    df = load_maxp(path)
    methods = methods or MAXP_METHODS
    fig, axes = plt.subplots(1, 2, figsize=(DCOL_W, 2.55),
                             constrained_layout=True)
    _maxp_sweep_panel(axes[0], df, dataset, "nPath", n_path,
                      "Budget", "probability", methods)
    _maxp_sweep_panel(axes[1], df, dataset, "nPath", n_path,
                      "Budget", "runtime", methods)
    return fig


def maxp_npath_figure(path: Path, dataset: str, budget: int = 100) -> plt.Figure:
    df = load_maxp(path)
    fig, axes = plt.subplots(1, 2, figsize=(DCOL_W, 2.55),
                             constrained_layout=True)
    _maxp_sweep_panel(axes[0], df, dataset, "Budget", budget,
                      "nPath", "probability", MAXP_METHODS)
    _maxp_sweep_panel(axes[1], df, dataset, "Budget", budget,
                      "nPath", "runtime", MAXP_METHODS)
    return fig


def mink_fixed_panel(ax: plt.Axes, df: pd.DataFrame, budget: int,
                     metric: str) -> None:
    sub = df[df["Budget"] == budget].copy()
    if sub.empty:
        raise ValueError(f"No minimum-telescope rows for D={budget}")
    datasets = sorted(sub["Dataset"].unique(), key=tile_count_from_name)
    x = np.arange(len(datasets))
    if metric == "routes":
        width = 0.84 / len(MINK_METHODS)
        for index, method in enumerate(MINK_METHODS):
            offsets = x - 0.42 + (index + 0.5) * width
            values, bounds = [], []
            for dataset in datasets:
                row = sub[(sub["Dataset"] == dataset)
                          & (sub["Method"] == method)]
                values.append(float(row["nPath"].iloc[0])
                              if not row.empty else np.nan)
                bounds.append(float(row["nPathBound"].iloc[0])
                              if not row.empty else np.nan)
            if method == "mink_rooted_ILP":
                ax.bar(offsets, bounds, width, color=MINK_PALETTE[method],
                       edgecolor="black", linewidth=0.25,
                       label=MINK_LABEL[method])
                for xpos, feasible, bound in zip(offsets, values, bounds):
                    gap = feasible - bound
                    if np.isfinite(gap) and gap > 0:
                        ax.bar(xpos, gap, width, bottom=bound,
                               facecolor="white",
                               edgecolor=MINK_PALETTE[method],
                               hatch="////", linewidth=0.45)
            else:
                ax.bar(offsets, values, width,
                       color=MINK_PALETTE[method], edgecolor="black",
                       linewidth=0.25, label=MINK_LABEL[method])
        ax.set_ylabel(r"$n_{\mathrm{path}}$")
        ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.set_ylim(bottom=0)
        ax.grid(axis="y", linestyle=":", alpha=0.55)
    elif metric == "runtime":
        for method in MINK_METHODS:
            values = []
            for dataset in datasets:
                row = sub[(sub["Dataset"] == dataset)
                          & (sub["Method"] == method)]
                values.append(float(row["PlanningTime"].iloc[0])
                              if not row.empty else np.nan)
            ax.plot(x, values, marker=MINK_MARKER[method],
                    color=MINK_PALETTE[method], label=MINK_LABEL[method])
        ax.set_yscale("log")
        ax.set_ylabel("Planning time (s)")
        ax.axhline(7200, color="grey", linestyle="--", linewidth=0.6)
        ax.text(x[-1], 7200, "120 min ILP cap", color="grey",
                fontsize=7, va="bottom", ha="right")
        ax.grid(True, which="both", linestyle=":", alpha=0.55)
    else:
        raise ValueError(metric)
    ax.set_xticks(x)
    ax.set_xticklabels([short_mink_dataset(dataset) for dataset in datasets],
                       rotation=25, ha="right")
    ax.set_xlabel("GW event (tile count)")
    ax.set_axisbelow(True)


def mink_cross_instance_figure(
    small_path: Path, large_path: Path, metric: str
) -> plt.Figure:
    small = load_mink(small_path)
    large = load_mink(large_path)
    fig, axes = plt.subplots(2, 1, figsize=(DCOL_W, 4.55))
    mink_fixed_panel(axes[0], small, budget=100, metric=metric)
    mink_fixed_panel(axes[1], large, budget=100, metric=metric)
    axes[0].text(0.01, 0.96, "Small maps", transform=axes[0].transAxes,
                 va="top", ha="left", fontsize=8)
    axes[1].text(0.01, 0.96, "Large maps", transform=axes[1].transAxes,
                 va="top", ha="left", fontsize=8)
    handles, labels = axes[0].get_legend_handles_labels()
    if metric == "routes":
        handles.append(Patch(facecolor="white",
                             edgecolor=MINK_PALETTE["mink_rooted_ILP"],
                             hatch="////", linewidth=0.5))
        labels.append("ILP gap to bound")
    fig.legend(handles, labels, loc="lower center",
               bbox_to_anchor=(0.5, 0.005), ncol=len(labels), fontsize=8)
    fig.tight_layout(rect=(0, 0.11, 1, 1), h_pad=0.7)
    return fig


def mink_budget_figure(path: Path, dataset: str) -> plt.Figure:
    df = load_mink(path)
    dataset = resolve_dataset(df, dataset)
    sub = df[df["Dataset"] == dataset].copy()
    x_values = sorted(sub["Budget"].unique())
    fig, axes = plt.subplots(1, 2, figsize=(DCOL_W, 2.55),
                             constrained_layout=True)
    for method in MINK_METHODS:
        method_rows = sub[sub["Method"] == method]
        xs, route_counts, bounds, times = [], [], [], []
        for value in x_values:
            row = method_rows[method_rows["Budget"] == value]
            if row.empty:
                continue
            xs.append(value)
            route_counts.append(float(row["nPath"].iloc[0]))
            bounds.append(float(row["nPathBound"].iloc[0]))
            times.append(float(row["PlanningTime"].iloc[0]))
        if not xs:
            continue
        axes[0].plot(xs, route_counts, marker=MINK_MARKER[method],
                     color=MINK_PALETTE[method], label=MINK_LABEL[method])
        if method == "mink_rooted_ILP":
            axes[0].plot(xs, bounds, linestyle="--", marker="",
                         color=MINK_PALETTE[method], linewidth=0.9,
                         label="R-ILP bound")
        axes[1].plot(xs, times, marker=MINK_MARKER[method],
                     color=MINK_PALETTE[method], label=MINK_LABEL[method])
    axes[0].set_ylabel(r"$n_{\mathrm{path}}$")
    axes[0].yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    axes[1].set_ylabel("Planning time (s)")
    axes[1].set_yscale("log")
    for ax in axes:
        ax.set_xticks(x_values)
        ax.set_xlabel("Deadline $D$ (s)")
        ax.grid(True, which="both", linestyle=":", alpha=0.55)
        ax.set_axisbelow(True)
        unique_legend(ax, loc="best", fontsize=7)
    return fig


## Route-map parsing and visualization helpers


In [ ]:
def wrap_lon_deg(values):
    values = np.asarray(values, dtype=float)
    return (values + 180.0) % 360.0 - 180.0


def radec_to_xyz(ra_rad, dec_rad):
    cos_dec = np.cos(dec_rad)
    return np.stack([
        cos_dec * np.cos(ra_rad),
        cos_dec * np.sin(ra_rad),
        np.sin(dec_rad),
    ], axis=-1)


def xyz_to_radec(xyz):
    x, y, z = xyz[..., 0], xyz[..., 1], xyz[..., 2]
    ra = np.arctan2(y, x) % (2 * np.pi)
    dec = np.arcsin(np.clip(z, -1.0, 1.0))
    return ra, dec


def slerp(u, v, count=12):
    u = u / np.linalg.norm(u)
    v = v / np.linalg.norm(v)
    dot = float(np.clip(np.dot(u, v), -1.0, 1.0))
    omega = math.acos(dot)
    if omega < 1e-10:
        return np.repeat(u[None, :], count, axis=0)
    t = np.linspace(0.0, 1.0, count)
    scale = math.sin(omega)
    points = (
        np.sin((1 - t) * omega)[:, None] * u[None, :]
        + np.sin(t * omega)[:, None] * v[None, :]
    ) / scale
    return points / np.linalg.norm(points, axis=1, keepdims=True)


def offset_sphere(ra0, dec0, position_angle, separation):
    sin_d1, cos_d1 = math.sin(dec0), math.cos(dec0)
    sin_sep, cos_sep = math.sin(separation), math.cos(separation)
    sin_d2 = (
        sin_d1 * cos_sep
        + cos_d1 * sin_sep * math.cos(position_angle)
    )
    dec2 = math.asin(np.clip(sin_d2, -1.0, 1.0))
    y = math.sin(position_angle) * sin_sep * cos_d1
    x = cos_sep - sin_d1 * math.sin(dec2)
    return (ra0 + math.atan2(y, x)) % (2 * math.pi), dec2


def tile_boundary_points(ra_deg, dec_deg, width_deg, height_deg,
                         edge_samples=8):
    ra0, dec0 = math.radians(ra_deg % 360.0), math.radians(dec_deg)
    corners = [
        (-width_deg / 2, -height_deg / 2),
        ( width_deg / 2, -height_deg / 2),
        ( width_deg / 2,  height_deg / 2),
        (-width_deg / 2,  height_deg / 2),
    ]
    corner_radec = []
    for east, north in corners:
        separation = math.radians(math.hypot(east, north))
        angle = math.atan2(math.radians(east), math.radians(north))
        corner_radec.append(offset_sphere(ra0, dec0, angle, separation))
    boundary = []
    for index in range(4):
        ra1, dec1 = corner_radec[index]
        ra2, dec2 = corner_radec[(index + 1) % 4]
        boundary.append(slerp(
            radec_to_xyz(ra1, dec1), radec_to_xyz(ra2, dec2),
            count=edge_samples,
        ))
    ra, dec = xyz_to_radec(np.vstack(boundary))
    return np.degrees(ra), np.degrees(dec)


def split_at_wrap(lon_deg, lat_deg, threshold=180.0):
    lon_deg = np.asarray(lon_deg, dtype=float)
    lat_deg = np.asarray(lat_deg, dtype=float)
    if not len(lon_deg):
        return []
    segments, xs, ys = [], [lon_deg[0]], [lat_deg[0]]
    for index in range(1, len(lon_deg)):
        if abs(lon_deg[index] - lon_deg[index - 1]) > threshold:
            if len(xs) > 1:
                segments.append((np.asarray(xs), np.asarray(ys)))
            xs, ys = [], []
        xs.append(lon_deg[index])
        ys.append(lat_deg[index])
    if len(xs) > 1:
        segments.append((np.asarray(xs), np.asarray(ys)))
    return segments


def sanitize_path_ids(values):
    cleaned = []
    for value in values:
        value = int(value)
        if value == -1:
            break
        if value < 0:
            continue
        if not cleaned or cleaned[-1] != value:
            cleaned.append(value)
    return cleaned


def parse_path_field(field):
    if field is None or (isinstance(field, float) and np.isnan(field)):
        return []
    values = []
    for token in str(field).strip().split():
        try:
            values.append(int(token))
        except ValueError:
            pass
    return sanitize_path_ids(values)


def read_paths_from_result_csv(
    path: Path,
    dataset: str,
    method: str,
    budget: int,
    n_path: int | None = None,
    row_index: int = 0,
) -> tuple[dict, list[list[int]]]:
    """Parse named or unnamed trailing path fields from a result row."""
    matches = []
    with path.open("r", encoding="utf-8", newline="") as stream:
        reader = csv.reader(stream)
        header = next(reader)
        columns = {name: index for index, name in enumerate(header)}
        for required in ["Method", "Dataset", "Budget", "nPath"]:
            if required not in columns:
                raise ValueError(f"Missing {required!r} in {path.name}")
        named_paths = [
            index for index, name in enumerate(header)
            if name.strip().lower().startswith("path")
        ]
        for raw in reader:
            if len(raw) < len(header):
                continue
            if normalize_dataset(raw[columns["Dataset"]]) != normalize_dataset(dataset):
                continue
            if raw[columns["Method"]] != method:
                continue
            try:
                if not np.isclose(float(raw[columns["Budget"]]), budget):
                    continue
                if n_path is not None and int(raw[columns["nPath"]]) != n_path:
                    continue
            except ValueError:
                continue
            matches.append((raw, named_paths, header))
    if not matches:
        raise ValueError(
            f"No result row for dataset={dataset}, method={method}, "
            f"budget={budget}, n_path={n_path} in {path.name}"
        )
    if row_index >= len(matches):
        raise IndexError(f"Only {len(matches)} rows matched; requested {row_index}")
    raw, named_paths, header = matches[row_index]
    metadata = {
        name: raw[index] if index < len(raw) else None
        for index, name in enumerate(header)
    }
    fields = ([raw[index] for index in named_paths if index < len(raw)]
              if named_paths else raw[len(header):])
    paths = [parsed for parsed in map(parse_path_field, fields) if parsed]
    if not paths:
        raise ValueError(f"No non-empty path fields found in {path.name}")
    return metadata, paths


def read_tile_centers(path: Path) -> dict[int, tuple[float, float]]:
    centers = {}
    with path.open("r", encoding="utf-8", newline="") as stream:
        reader = csv.reader(stream)
        header = next(reader)
        columns = {name.strip().upper(): index
                   for index, name in enumerate(header)}
        for required in ["ID", "RA", "DEC"]:
            if required not in columns:
                raise ValueError(f"Missing {required!r} in {path.name}")
        for row in reader:
            if not row:
                continue
            centers[int(row[columns["ID"]])] = (
                float(row[columns["RA"]]), float(row[columns["DEC"]])
            )
    return centers


def read_likelihood_points(path: Path, coverage: float = 0.999):
    lon, lat, probability = [], [], []
    with path.open("r", encoding="utf-8") as stream:
        for line in stream:
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            parts = stripped.split()
            if len(parts) < 4:
                continue
            try:
                dec, ra, prob = float(parts[1]), float(parts[2]), float(parts[3])
            except ValueError:
                continue
            if np.isfinite(prob) and prob >= 0:
                lon.append(ra); lat.append(dec); probability.append(prob)
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)
    probability = np.asarray(probability, dtype=float)
    if not len(probability) or probability.sum() <= 0:
        raise ValueError(f"No positive likelihood values in {path.name}")
    order = np.argsort(probability)[::-1]
    count = np.searchsorted(
        np.cumsum(probability[order]), coverage * probability.sum(), side="left"
    ) + 1
    kept = order[:count]
    floor = probability[kept].min()
    shown = np.full_like(probability, floor)
    shown[kept] = probability[kept]
    return lon, lat, shown


def route_context(tile_path: Path, map_path: Path):
    lon, lat, probability = read_likelihood_points(map_path)
    centers = read_tile_centers(tile_path)
    return {
        "lon": lon,
        "lat": lat,
        "probability": probability,
        "centers": centers,
    }


def path_centers(paths: list[list[int]], centers):
    result = []
    missing = []
    for path in paths:
        one = []
        for tile_id in path:
            if tile_id in centers:
                one.append((tile_id, centers[tile_id]))
            else:
                missing.append(tile_id)
        if one:
            result.append(one)
    if missing:
        print(f"[route map] skipped {len(missing)} unknown tile id(s)")
    return result


def draw_route_panel(
    ax: plt.Axes,
    context: dict,
    paths: list[list[int]],
    title: str,
    width_deg: float = 6.9,
    height_deg: float = 6.9,
    draw_all_tiles: bool = True,
) -> None:
    probability = context["probability"]
    ax.scatter(
        np.radians(wrap_lon_deg(context["lon"])),
        np.radians(context["lat"]),
        c=np.log10(np.maximum(probability, 1e-30)),
        cmap="viridis", s=2.0, alpha=0.78, rasterized=True, zorder=1,
    )
    if draw_all_tiles:
        for _, (ra, dec) in context["centers"].items():
            boundary_ra, boundary_dec = tile_boundary_points(
                ra, dec, width_deg, height_deg, edge_samples=5
            )
            for xs, ys in split_at_wrap(wrap_lon_deg(boundary_ra), boundary_dec):
                ax.plot(np.radians(xs), np.radians(ys), color="0.72",
                        linewidth=0.25, alpha=0.45, zorder=2)

    converted = path_centers(paths, context["centers"])
    for index, route in enumerate(converted):
        color = ROUTE_COLORS[index % len(ROUTE_COLORS)]
        route_ra = np.asarray([center[0] for _, center in route])
        route_dec = np.asarray([center[1] for _, center in route])
        for xs, ys in split_at_wrap(wrap_lon_deg(route_ra), route_dec):
            ax.plot(np.radians(xs), np.radians(ys), "-o", color=color,
                    linewidth=1.4, markersize=1.8, zorder=4)
        start_ra, start_dec = route[0][1]
        ax.scatter([math.radians(float(wrap_lon_deg(start_ra)))],
                   [math.radians(start_dec)], s=18, color=color,
                   edgecolor="white", linewidth=0.4, zorder=5)
        ax.text(math.radians(float(wrap_lon_deg(start_ra))),
                math.radians(start_dec), f"R{index}", color="white",
                fontsize=6.5, ha="left", va="bottom", zorder=6)
    ax.grid(True, alpha=0.25, linewidth=0.4)
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.set_title(title, fontsize=9, pad=4)


def load_route_paths(result_path: Path, dataset: str, method: str,
                     budget: int, n_path: int | None = None):
    _, paths = read_paths_from_result_csv(
        result_path, dataset=dataset, method=method,
        budget=budget, n_path=n_path,
    )
    return paths


def figure_4_routes() -> plt.Figure:
    context = route_context(FILES["tiling_6p9"], FILES["map_200220_199"])
    methods = [
        ("greedy+annealing", "(a) Greedy+SA"),
        ("greedy+ilp", "(b) Greedy+ILP"),
        ("top_annealing", "(c) TOP-SA"),
        ("top_ILP", "(d) TOP-ILP"),
    ]
    fig, axes = plt.subplots(2, 2, figsize=(DCOL_W, 4.1),
                             subplot_kw={"projection": "mollweide"},
                             constrained_layout=True)
    for ax, (method, label) in zip(axes.flat, methods):
        paths = load_route_paths(
            FILES["maxp_small"], DATASET_200220,
            method, budget=100, n_path=4,
        )
        draw_route_panel(ax, context, paths, label,
                         draw_all_tiles=DRAW_ALL_TILES)
    return fig


def figure_10_routes() -> plt.Figure:
    context = route_context(FILES["tiling_6p9"], FILES["map_200216_129"])
    methods = [
        ("mink_rooted_ILP", "(a) R-ILP"),
        ("mink_rooted_greedy", "(b) R-Greedy"),
        ("mink_rooted_annealing", "(c) R-SA"),
    ]
    fig = plt.figure(figsize=(DCOL_W, 4.1), constrained_layout=True)
    grid = GridSpec(2, 4, figure=fig)
    axes = [
        fig.add_subplot(grid[0, :2], projection="mollweide"),
        fig.add_subplot(grid[0, 2:], projection="mollweide"),
        fig.add_subplot(grid[1, 1:3], projection="mollweide"),
    ]
    for ax, (method, label) in zip(axes, methods):
        paths = load_route_paths(
            FILES["mink_small"], DATASET_200216,
            method, budget=100,
        )
        draw_route_panel(ax, context, paths, label,
                         draw_all_tiles=DRAW_ALL_TILES)
    return fig


def tile_grid_coordinates(tile_centers, width_deg=6.9, height_deg=6.9,
                          edge_samples=5):
    """Combine tile footprints into one Plotly line trace."""
    grid_lon, grid_lat = [], []
    for _, (ra, dec) in tile_centers:
        boundary_ra, boundary_dec = tile_boundary_points(
            ra, dec, width_deg, height_deg, edge_samples=edge_samples,
        )
        for xs, ys in split_at_wrap(wrap_lon_deg(boundary_ra), boundary_dec):
            grid_lon.extend(xs.tolist())
            grid_lat.extend(ys.tolist())
            grid_lon.append(None)
            grid_lat.append(None)
    return grid_lon, grid_lat


def add_tile_grid_trace(fig, tile_centers, color, width,
                        width_deg=6.9, height_deg=6.9):
    grid_lon, grid_lat = tile_grid_coordinates(
        tile_centers, width_deg=width_deg, height_deg=height_deg,
    )
    if grid_lon:
        fig.add_trace(go.Scattergeo(
            lon=grid_lon,
            lat=grid_lat,
            mode="lines",
            line={"width": width, "color": color},
            showlegend=False,
            hoverinfo="skip",
        ))


def interactive_route_view(context: dict, paths: list[list[int]], title: str):
    """Build a directly labeled, legend-free Plotly Mollweide viewer."""
    fig = go.Figure()
    fig.add_trace(go.Scattergeo(
        lon=wrap_lon_deg(context["lon"]),
        lat=context["lat"],
        mode="markers",
        marker={
            "size": 5,
            "color": np.log10(np.maximum(context["probability"], 1e-30)),
            "colorscale": "Viridis",
            "showscale": False,
        },
        showlegend=False,
        hoverinfo="skip",
    ))
    converted_paths = path_centers(paths, context["centers"])
    if DRAW_ALL_TILES:
        add_tile_grid_trace(
            fig,
            context["centers"].items(),
            color="rgba(170,170,170,0.28)",
            width=0.7,
        )
    visited_tiles = []
    seen_tile_ids = set()
    for route in converted_paths:
        for tile_id, center in route:
            if tile_id not in seen_tile_ids:
                seen_tile_ids.add(tile_id)
                visited_tiles.append((tile_id, center))
    add_tile_grid_trace(
        fig,
        visited_tiles,
        color="rgba(80,80,80,0.60)",
        width=1.2,
    )
    for index, route in enumerate(converted_paths):
        ra = np.asarray([center[0] for _, center in route])
        dec = np.asarray([center[1] for _, center in route])
        ids = [tile_id for tile_id, _ in route]
        color = ROUTE_COLORS[index % len(ROUTE_COLORS)]
        hover_text = [
            f"route {index}, tile {tile_id}, order {order}"
            for order, tile_id in enumerate(ids)
        ]
        fig.add_trace(go.Scattergeo(
            lon=wrap_lon_deg(ra),
            lat=dec,
            mode="lines+markers",
            line={"width": 3, "color": color},
            marker={"size": 5, "color": color},
            showlegend=False,
            hovertext=hover_text,
            hovertemplate="%{hovertext}<extra></extra>",
        ))
        fig.add_trace(go.Scattergeo(
            lon=[float(wrap_lon_deg(ra[0]))],
            lat=[float(dec[0])],
            mode="markers+text",
            text=[f"R{index}"],
            textposition="top center",
            textfont={"size": 16, "color": "white"},
            marker={"size": 10, "color": color, "symbol": "circle"},
            showlegend=False,
            hovertext=[f"route {index} start, tile {ids[0]}"],
            hovertemplate="%{hovertext}<extra></extra>",
        ))
    fig.update_layout(
        title=title,
        showlegend=False,
        geo={
            "projection_type": "mollweide",
            "showland": False,
            "showcoastlines": False,
            "showframe": True,
            "lataxis_showgrid": True,
            "lonaxis_showgrid": True,
        },
        height=550,
    )
    return fig


## Figure 2 - Fixed-K detection probability across maps

Detection probability for $K=4$ telescopes and deadline $D=100$ s. Small-map instances are shown above large-map instances; hatched ILP extensions show the gap from the feasible solution to the solver bound.


In [ ]:
build_or_skip(
    2, "fig02_maxp_detection_probability",
    [FILES["maxp_small"], FILES["maxp_large"]],
    lambda: maxp_cross_instance_figure(
        FILES["maxp_small"], FILES["maxp_large"], "probability"
    ),
)


## Figure 3 - Fixed-K planning time across maps

Planning time for $K=4$ telescopes and deadline $D=100$ s, with a logarithmic time axis.


In [ ]:
build_or_skip(
    3, "fig03_maxp_planning_time",
    [FILES["maxp_small"], FILES["maxp_large"]],
    lambda: maxp_cross_instance_figure(
        FILES["maxp_small"], FILES["maxp_large"], "runtime"
    ),
)


## Figure 4 - Fixed-K routes on map #200220

Routes returned by Greedy+SA, Greedy+ILP, TOP-SA, and TOP-ILP for the representative small-map instance 200220(199), $K=4$, and $D=100$ s.


In [ ]:
build_or_skip(
    4, "fig04_maxp_routes_200220",
    [FILES["tiling_6p9"], FILES["map_200220_199"], FILES["maxp_small"]],
    figure_4_routes,
)

if RUN_INTERACTIVE and all(path.exists() for path in [
    FILES["tiling_6p9"], FILES["map_200220_199"], FILES["maxp_small"]
]):
    _context = route_context(FILES["tiling_6p9"], FILES["map_200220_199"])
    for _method, _label in [
        ("greedy+annealing", "Greedy+SA"),
        ("greedy+ilp", "Greedy+ILP"),
        ("top_annealing", "TOP-SA"),
        ("top_ILP", "TOP-ILP"),
    ]:
        _paths = load_route_paths(
            FILES["maxp_small"], DATASET_200220,
            _method, budget=100, n_path=4,
        )
        interactive_route_view(
            _context, _paths, f"Figure 4: {_label}"
        ).show()


## Figure 5 - Effect of deadline on large map #191113

Detection probability and planning time as deadline $D$ varies on 191113(952), with $K=4$ telescopes.


In [ ]:
build_or_skip(
    5, "fig05_maxp_deadline_191113",
    [FILES["maxp_large_budget"]],
    lambda: maxp_budget_figure(
        FILES["maxp_large_budget"], DATASET_LARGE, n_path=4
    ),
)


## Figure 6 - Effect of deadline on very large map #200105

Detection probability and planning time on the 11,678-tile instance, comparing the two practical simulated-annealing methods for $K=4$.


In [ ]:
build_or_skip(
    6, "fig06_maxp_deadline_200105_11678",
    [FILES["maxp_very_large_budget"]],
    lambda: maxp_budget_figure(
        FILES["maxp_very_large_budget"], DATASET_VERY_LARGE,
        n_path=4,
        methods=["greedy+annealing", "top_annealing"],
    ),
)


## Figure 7 - Effect of telescope count on large map #191113

Detection probability and planning time as the number of telescopes $K$ varies, with deadline $D=100$ s.


In [ ]:
build_or_skip(
    7, "fig07_maxp_telescope_count_191113",
    [FILES["maxp_large_npath"]],
    lambda: maxp_npath_figure(
        FILES["maxp_large_npath"], DATASET_LARGE, budget=100
    ),
)


## Figure 8 - Minimum telescope demand across maps

Number of rooted routes returned at deadline $D=100$ s. The solid R-ILP portion is the lower bound and its hatched extension reaches the best feasible R-ILP solution.


In [ ]:
build_or_skip(
    8, "fig08_mink_route_count",
    [FILES["mink_small"], FILES["mink_large"]],
    lambda: mink_cross_instance_figure(
        FILES["mink_small"], FILES["mink_large"], "routes"
    ),
)


## Figure 9 - Minimum-telescope planning time across maps

Planning time for the minimum-telescope-demand objective at deadline $D=100$ s, with a logarithmic time axis.


In [ ]:
build_or_skip(
    9, "fig09_mink_planning_time",
    [FILES["mink_small"], FILES["mink_large"]],
    lambda: mink_cross_instance_figure(
        FILES["mink_small"], FILES["mink_large"], "runtime"
    ),
)


## Figure 10 - Minimum-telescope routes on map #200216

Routes returned by R-ILP, R-Greedy, and R-SA for representative small-map instance 200216(129) at deadline $D=100$ s.


In [ ]:
build_or_skip(
    10, "fig10_mink_routes_200216",
    [FILES["tiling_6p9"], FILES["map_200216_129"], FILES["mink_small"]],
    figure_10_routes,
)

if RUN_INTERACTIVE and all(path.exists() for path in [
    FILES["tiling_6p9"], FILES["map_200216_129"], FILES["mink_small"]
]):
    _context = route_context(FILES["tiling_6p9"], FILES["map_200216_129"])
    for _method, _label in [
        ("mink_rooted_ILP", "R-ILP"),
        ("mink_rooted_greedy", "R-Greedy"),
        ("mink_rooted_annealing", "R-SA"),
    ]:
        _paths = load_route_paths(
            FILES["mink_small"], DATASET_200216,
            _method, budget=100,
        )
        interactive_route_view(
            _context, _paths, f"Figure 10: {_label}"
        ).show()


## Figure 11 - Effect of deadline on minimum telescope demand

Number of rooted routes and planning time as deadline $D$ varies on large-map instance 191113(952). Smaller route counts are better.


In [ ]:
build_or_skip(
    11, "fig11_mink_deadline_191113",
    [FILES["mink_large_budget"]],
    lambda: mink_budget_figure(
        FILES["mink_large_budget"], DATASET_LARGE
    ),
)
